# Example: choosing the number of components

The families of the components are known — normal — but their number is not. A common recipe is to fit a mixture for
every candidate `k` and choose the smallest Bayesian information criterion,
`BIC = -2 ℓ + p ln n`, where `p` counts the free parameters and the weights. Every fit needs a start: here each `k`
gets a multistart of refined runs, with its own child of one seed so that the whole search is reproducible.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

from pysatl_mpest.initializers import ClusterInitializer, KMeansPartitioner, MultiStartInitializer, RefiningInitializer

## Data

In [ ]:
truth = MixtureModel([normal(mu=-6, sigma=1), normal(mu=0, sigma=1.5), normal(mu=5, sigma=1)], weights=[0.3, 0.4, 0.3])
X = truth.generate(size=900, random_state=7)

## A fit for every k

In [ ]:
initializer = MultiStartInitializer(RefiningInitializer(ClusterInitializer(KMeansPartitioner()), em(50)), n_init=5)
candidates = range(1, 7)
fits, bic = {}, {}
for k, child in zip(candidates, np.random.default_rng(42).spawn(len(candidates))):
    fits[k] = initializer.initialize(X, [normal] * k, random_state=child)
    p = sum(len(component.parameters) for component in fits[k]) + k - 1
    bic[k] = -2 * fits[k].loglikelihood(X) + p * np.log(len(X))
    print(f"k={k}  log-likelihood {fits[k].loglikelihood(X):9.1f}  BIC {bic[k]:9.1f}")
chosen = min(bic, key=bic.get)
print("chosen k:", chosen)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 3.5))
axes[0].plot(list(bic), list(bic.values()), "o-")
axes[0].axvline(chosen, color="tab:red", linestyle="--")
axes[0].set_xlabel("k")
axes[0].set_ylabel("BIC")
plot_mixtures(axes[1], X, {f"k={chosen}": fits[chosen], "truth": truth})
plt.show()

assert chosen == 3

## Takeaway

The log-likelihood always grows with `k`; BIC pays for every extra parameter and stops at three. Giving every `k` its
own child of one generator keeps the search reproducible, and a different seed changes all fits at once.